In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
from mex.task import RepetitionDecreaseTask, SegmentBoundaryTask, IDyOMConsistencyTask, NeuralEncodingTask
from mex.metadata import SessionIDyOMMetaData, SalamiMetaData, SongMetaData
from mex.dataset_provider import DatasetProvider
from mex.baseline.audioic import AudioIC

/home/mathias/.conda/envs/mex/lib/python3.15/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/mathias/.conda/envs/mex/lib/python3.15/site-packages/torch/jit/_script.py:1485: FutureWarning: `torch.jit.script` is not supported in Python 3.14+ and may break. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


Typical use of MEX:
- Preprocess task metadata or load from task
- Estimate MIDMs on an audio dataset
- Run the task

# Task metadata

load dataset provider, to get resources from huggingface hub.

In [4]:
dataset_provider = DatasetProvider(
    cache_dir=None, # NOTE: will cache datasets in platform specific location, e.g. ~/.cache/mex on linux
    hf_root="MusicPerceptModeling/mex",
)
salami_metadata = SalamiMetaData(
    salami_root="/dir/to/salami"
)

session_idyom_metadata = SessionIDyOMMetaData(
    idyom_dat_dir="/dir/to/idyom.dat",
    midi_dir="/dir/to/midi/files"
)

neural_encoding_metadata = SongMetaData(
    mat_dir="/dir/to/mat",
)

## Preprocess
(skip if using already cached datasets, or loading from hub)

preprocess datasets and store cached versions in `dataset_provider.cache_dir`

In [ ]:
salami_metadata.preprocess()
session_idyom_metadata.preprocess()

## Load precomputed metadata datasets

In [5]:
# get datasets and parse
salami_metadata_ds = salami_metadata.load(dataset_provider)
session_metdata_ds = session_idyom_metadata.load(dataset_provider)
song_metdata_ds = neural_encoding_metadata.load(dataset_provider)
# or from non_standard location, e.g. local or hfhub
session_metdata_ds_2 = dataset_provider.get_ds("session_idyom")
assert session_metdata_ds_2.features == session_metdata_ds.features

# Running tasks

## IDyOM consistency

In [5]:
# optionally filter dataset (for faster testing)
n_audios = 20
session_metdata_ds_small = session_metdata_ds.select(range(n_audios))

In [6]:
# load baseline model
baseline = AudioIC(device='cpu')

In [ ]:
ic_ds = baseline.ic(session_metdata_ds_small)

Computing IC: 100%|██████████| 20/20 [05:21<00:00, 16.09s/it]


In [27]:
ic_ds.features

{'id': Value('int64'),
 'instrument': Value('string'),
 'ic': List(Value('float32'))}

In [ ]:
# load only the relevant columns for idyom IC correlation from session_ds (idyom_ic)
idyom_ds = session_metdata_ds_small.select_columns(["id", "onset",  "idyom_ic"]) 
idyom_consistency_task = IDyOMConsistencyTask(
    periode_frames=AudioIC.periode_frames(),
)
res = idyom_consistency_task.run_task(
    midm=ic_ds,
    metadata=session_metdata_ds,
)
print(res)

             instrument label    r_val  p_val    idyom_midm
   Acoustic Grand Piano    ic 0.504193    0.0      idyom_ic
Acoustic Guitar (nylon)    ic 0.498678    0.0      idyom_ic
        Pad 1 (new age)    ic 0.534058    0.0      idyom_ic
            Synth Voice    ic 0.479777    0.0      idyom_ic
                Trumpet    ic 0.543278    0.0      idyom_ic
               combined    ic 0.470411    0.0      idyom_ic
   Acoustic Grand Piano    ic 0.439009    0.0 idyom_entropy
Acoustic Guitar (nylon)    ic 0.427731    0.0 idyom_entropy
        Pad 1 (new age)    ic 0.431896    0.0 idyom_entropy
            Synth Voice    ic 0.373372    0.0 idyom_entropy
                Trumpet    ic 0.457446    0.0 idyom_entropy
               combined    ic 0.389537    0.0 idyom_entropy


In [ ]:
# running the experiment on all values in the dataset
ic_ds = dataset_provider.get_ds("cache/metrics/session_audioic_ic", use_cache=False)
idyom_ds = session_metdata_ds.select_columns(["id", "onset",  "idyom_ic"]) 
# idyom_ds = idyom_ds.rename_column("idyom_ic", "idyom_val")
result = idyom_consistency_task.run_task(
    midm=ic_ds,
    metadata=session_metdata_ds,
)
print(result)

             instrument label    r_val  p_val    idyom_midm
   Acoustic Grand Piano    ic 0.504193    0.0      idyom_ic
Acoustic Guitar (nylon)    ic 0.498678    0.0      idyom_ic
        Pad 1 (new age)    ic 0.534058    0.0      idyom_ic
            Synth Voice    ic 0.479777    0.0      idyom_ic
                Trumpet    ic 0.543278    0.0      idyom_ic
               combined    ic 0.470411    0.0      idyom_ic
   Acoustic Grand Piano    ic 0.439009    0.0 idyom_entropy
Acoustic Guitar (nylon)    ic 0.427731    0.0 idyom_entropy
        Pad 1 (new age)    ic 0.431896    0.0 idyom_entropy
            Synth Voice    ic 0.373372    0.0 idyom_entropy
                Trumpet    ic 0.457446    0.0 idyom_entropy
               combined    ic 0.389537    0.0 idyom_entropy


## Repetition decrease

In [8]:

# Get precomputed IC values for salami dataset
salami_ic = dataset_provider.get_ds("cache/metrics/salami_audioic_ic/", use_cache=False).remove_columns("Time")
# Get salami segment annotation metadata from local disk
repetion_decrease_task = RepetitionDecreaseTask(
    periode_frames = AudioIC.periode_frames(),
    time_offset_frame =  1 # for IC 1 for entropy 0
)
results = repetion_decrease_task.run_task(
    midm=salami_ic,
    metadata=salami_metadata_ds,
)
print(results)

Using the latest cached version of the dataset since MusicPerceptModeling/mex couldn't be found on the Hugging Face Hub
Map: 100%|██████████| 1358/1358 [00:07<00:00, 181.67 examples/s]


midm_column  mode  seg_name      mean      p_value
         IC   all functions -0.047633 9.290729e-39
         IC   all lowercase -0.029830 2.409445e-72
         IC   all uppercase -0.054142 1.600254e-49
         IC first functions -0.082067 5.025470e-45
         IC first lowercase -0.075825 6.817193e-88
         IC first uppercase -0.053426 2.408463e-19


## Segment Boundary

In [9]:
segment_boundary_task = SegmentBoundaryTask(
    periode_frames = AudioIC.periode_frames(),
    time_offset_frame =  1 # for IC 1 for entropy 0
)
results = segment_boundary_task.run_task(
    midm=salami_ic,
    metadata=salami_metadata_ds,
)
print(results)

Map: 100%|██████████| 1358/1358 [00:05<00:00, 262.77 examples/s]


hierarchy_level      repetition_mode midm peak_picking_kwargs  precision   recall  f_measure
      functions                  all   IC        {"std": 5.0}   0.088948 0.545451   0.152954
      functions               change   IC        {"std": 5.0}   0.071994 0.529113   0.126743
      functions only_first_occurence   IC        {"std": 5.0}   0.039375 0.509501   0.073101
      lowercase                  all   IC        {"std": 5.0}   0.295140 0.567054   0.388220
      lowercase               change   IC        {"std": 5.0}   0.207167 0.571179   0.304054
      lowercase only_first_occurence   IC        {"std": 5.0}   0.075442 0.576996   0.133437
      uppercase                  all   IC        {"std": 5.0}   0.085152 0.551562   0.147529
      uppercase               change   IC        {"std": 5.0}   0.067990 0.537783   0.120717
      uppercase only_first_occurence   IC        {"std": 5.0}   0.036908 0.525581   0.068973


# Neural Encoding

In [ ]:
# calculate IC with baseline model
song_ds = dataset_provider.get_ds("song_audio")
ic_song = baseline.ic(song_ds)

In [ ]:
# or get precomputed IC values
ic_song = dataset_provider.get_ds("song_pa-audioic_ic-0.0-0.3-0.5")

In [ ]:
neural_encoding_task = NeuralEncodingTask(
    midm_column="ic_noise_0_0",
    ic_dt=AudioIC.periode_frames(),
)
results = neural_encoding_task.run_task(
    midm=ic_song,
    metadata=song_metdata_ds,
)
print(results)

[Sub1] Merged 14 valid epochs at 64.0 Hz.
Fitting Baseline Model (Energy only)...
Best energy-only lambda: 0.1
[Sub10] Merged 18 valid epochs at 64.0 Hz.
Fitting Baseline Model (Energy only)...
Best energy-only lambda: 10.0
[Sub11] Merged 18 valid epochs at 64.0 Hz.
Fitting Baseline Model (Energy only)...
Best energy-only lambda: 1.0
[Sub12] Merged 18 valid epochs at 64.0 Hz.
Fitting Baseline Model (Energy only)...
Best energy-only lambda: 10.0
[Sub13] Merged 18 valid epochs at 64.0 Hz.
Fitting Baseline Model (Energy only)...
Best energy-only lambda: 1.0
[Sub14] Merged 18 valid epochs at 64.0 Hz.
Fitting Baseline Model (Energy only)...
Best energy-only lambda: 1000.0
[Sub15] Merged 18 valid epochs at 64.0 Hz.
Fitting Baseline Model (Energy only)...
Best energy-only lambda: 100.0
[Sub16] Merged 13 valid epochs at 64.0 Hz.
Fitting Baseline Model (Energy only)...
Best energy-only lambda: 10.0
[Sub17] Merged 18 valid epochs at 64.0 Hz.
Fitting Baseline Model (Energy only)...
Best energy-on